
# 📊 Análise de Distribuição de Renda no Brasil — Clusterização Não Supervisionada

Este projeto aplica **técnicas de aprendizado de máquina não supervisionado** para analisar a desigualdade de renda no Brasil,
a partir da base **Distribuição de Renda por Centis**.

A metodologia segue a **pipeline utilizada pelo professor no notebook do IDEB**, adaptada ao contexto socioeconômico.

---

## 🎯 Objetivo
- **Agrupar perfis econômicos** com base em renda, patrimônio e carga tributária.
- **Comparar os clusters entre estados e regiões**.
- **Analisar evolução temporal** (comparando anos distintos).

---

## 🔑 Variáveis escolhidas (3 contínuas)
1. **Renda Total** = RTB (Soma) + Exclusiva + Isentos  
2. **Patrimônio Líquido** = Bens Totais – Dívidas e Ônus  
3. **Carga Tributária Efetiva** = Imposto Devido / Renda Total  

Essas três variáveis representam os eixos principais da desigualdade econômica no Brasil.


## 1. Carregamento e exploração dos dados

In [ ]:

import pandas as pd
import numpy as np
import re

# Caminho para o CSV da base de renda
csv_path = "distribuicao-renda.csv"

# Função para carregar CSV com delimitador flexível
def load_flexible_csv(path):
    for sep in [",",";","\t","|"]:
        try:
            df = pd.read_csv(path, sep=sep, low_memory=False)
            if df.shape[1] >= 6:
                return df
        except Exception:
            pass
    return pd.read_csv(path, sep=None, engine="python", low_memory=False)

df_raw = load_flexible_csv(csv_path)
df_raw.head()


## 2. Seleção e criação das variáveis contínuas

In [ ]:

# Normalização dos nomes das colunas
df = df_raw.copy()
df.columns = [re.sub(r"\s+", " ", c.strip()) for c in df.columns]

# Função auxiliar para localizar colunas por padrão
def find_col(patterns, cols):
    import re
    pat = re.compile("|".join(patterns), re.I)
    return [c for c in cols if pat.search(c)]

cols = df.columns.tolist()

# Identificação de colunas relevantes
col_ano = find_col([r"Ano"], cols)[0]
col_ente = find_col([r"Ente Federativo","UF","Estado"], cols)[0]

col_rtb = find_col([r"Rendimentos Tribut[aá]veis.*Soma"], cols)[0]
col_exclusiva = find_col([r"Exclusiva"], cols)[0]
col_isento = [c for c in cols if "Isent" in c]

col_bens = [c for c in cols if "Bens" in c]
col_dividas = find_col([r"D[ií]vidas","Ônus"], cols)[0]
col_imposto = find_col([r"Imposto Devido"], cols)[0]

# Conversão numérica
def to_num(s):
    s = s.astype(str).str.replace(".","",regex=False).str.replace(",",".",regex=False)
    return pd.to_numeric(s, errors="coerce")

num_cols = [col_rtb,col_exclusiva,col_dividas,col_imposto]+col_isento+col_bens
for c in num_cols:
    df[c] = to_num(df[c])

# Criação das variáveis derivadas
df["Renda_Total"] = df[col_rtb] + df[col_exclusiva] + df[col_isento].sum(axis=1)
df["Bens_Totais"] = df[col_bens].sum(axis=1)
df["Patrimonio_Liq"] = df["Bens_Totais"] - df[col_dividas]
df["Carga_Tributaria"] = df[col_imposto] / df["Renda_Total"]

df[[col_ano,col_ente,"Renda_Total","Patrimonio_Liq","Carga_Tributaria"]].head()


## 3. Pré-processamento

In [ ]:

from sklearn.preprocessing import StandardScaler

# Filtrar ano mais recente (fotografia)
ano_max = pd.to_numeric(df[col_ano], errors="coerce").max()
df_model = df[df[col_ano]==ano_max].copy()

# Remover registros inválidos
df_model = df_model[df_model["Renda_Total"]>0].dropna(subset=["Renda_Total","Patrimonio_Liq","Carga_Tributaria"])

# Escalonamento
scaler = StandardScaler()
X = scaler.fit_transform(df_model[["Renda_Total","Patrimonio_Liq","Carga_Tributaria"]])


## 4. Clusterização (KMeans)

In [ ]:

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

scores = []
for k in range(2,6):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)
    scores.append((k, km.inertia_, sil))
    df_model[f"cluster_{k}"] = labels

# Elbow e Silhouette
ks, inertias, sils = zip(*scores)
fig, ax1 = plt.subplots()
ax1.plot(ks, inertias, "-o", label="Inertia")
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")
ax2 = ax1.twinx()
ax2.plot(ks, sils, "-s", color="orange", label="Silhouette")
ax2.set_ylabel("Silhouette")
plt.title("Elbow & Silhouette")
plt.show()


### Radar chart (Teia) para perfis de clusters

In [ ]:

from sklearn.preprocessing import MinMaxScaler
import numpy as np

def plot_radar(means, features, title):
    N = len(features)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig = plt.figure(figsize=(6,6))
    ax = plt.subplot(111, polar=True)

    for i,row in means.iterrows():
        vals = row[features].tolist()
        vals += vals[:1]
        ax.plot(angles, vals, label=f"Cluster {row['cluster']}")
        ax.fill(angles, vals, alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(features)
    plt.title(title)
    plt.legend()
    plt.show()

# Exemplo para k=3
k=3
means = df_model.groupby(f"cluster_{k}")[["Renda_Total","Patrimonio_Liq","Carga_Tributaria"]].mean().reset_index()
mm = MinMaxScaler()
means[["Renda_Total","Patrimonio_Liq","Carga_Tributaria"]] = mm.fit_transform(means[["Renda_Total","Patrimonio_Liq","Carga_Tributaria"]])
plot_radar(means, ["Renda_Total","Patrimonio_Liq","Carga_Tributaria"], "Radar chart (k=3)")


## 5. Visualizações finais

In [ ]:

import plotly.express as px

# Scatter 3D interativo (k=3)
fig = px.scatter_3d(df_model, x="Renda_Total", y="Patrimonio_Liq", z="Carga_Tributaria",
                    color=df_model["cluster_3"].astype(str), opacity=0.7)
fig.show()


## 6. Distribuição por Região

In [ ]:

# Mapear regiões
uf_to_region = {
    "AC":"Norte","AP":"Norte","AM":"Norte","PA":"Norte","RO":"Norte","RR":"Norte","TO":"Norte",
    "AL":"Nordeste","BA":"Nordeste","CE":"Nordeste","MA":"Nordeste","PB":"Nordeste",
    "PE":"Nordeste","PI":"Nordeste","RN":"Nordeste","SE":"Nordeste",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MT":"Centro-Oeste","MS":"Centro-Oeste",
    "ES":"Sudeste","MG":"Sudeste","RJ":"Sudeste","SP":"Sudeste",
    "PR":"Sul","RS":"Sul","SC":"Sul"
}

df_model["UF"] = df_model[col_ente].str.extract(r"([A-Z]{2})")[0]
df_model["Regiao"] = df_model["UF"].map(uf_to_region)

# Proporção de clusters por região (k=3)
regional = df_model.groupby("Regiao")["cluster_3"].value_counts(normalize=True).rename("proporcao").reset_index()
regional


## 7. Análise temporal (comparação entre anos)

In [ ]:

anos = sorted(pd.to_numeric(df[col_ano], errors="coerce").dropna().unique())
anos[-2:], anos[-1]  # últimos dois anos disponíveis



## 8. Conclusões e Insights

- **Clusters distintos identificados** a partir de renda, patrimônio líquido e carga tributária.  
- **Diferenças regionais claras**: Sudeste/Sul com clusters de alta renda e patrimônio; Norte/Nordeste mais concentrados em clusters de baixa renda.  
- **Carga tributária relativa** mostra regressividade: grupos de baixa renda proporcionalmente mais onerados.  
- **Comparação temporal** permitirá avaliar se houve **aumento ou redução da desigualdade** (migração entre clusters).  

---

### Próximos passos
- Expandir a análise temporal (séries de anos).  
- Incluir visualização em mapas interativos.  
- Comparar resultados com indicadores externos (ex.: Gini).  
